# Sesión 5 — Backup-as-Code
**Módulo 9 (GitHub) + Módulo 10 (Backups y Recuperación) fusionados**

Curso GCP + Vertex AI para People Analytics · Imagina Formación

---

## Qué construimos hoy

Un repo `people-analytics-ops` (local en este notebook, opcionalmente subible a GitHub) con:

- `backups/sql/` — scripts SQL versionados que crean snapshots reales sobre tablas de S3 (`predictions_retention.retention_actions`) y S1+S2 (`silver_personio.dim_employee`).
- `backups/python/` — `export_to_gcs.py` (Parquet+Snappy) y `restore_test.py` (job mensual).
- `backups/policies/RETENTION.md` — política firmable por DPO.
- `.github/workflows/` — `validate-backups.yml` (PR gate) y `scheduled-snapshot.yml` (cron diario).
- `.github/CODEOWNERS` + PR template.

Y al final lo probamos: creamos una tabla, la **borramos**, y la **recuperamos por 3 caminos**: time travel (`copy_table` con `@ms`), snapshot (`CLONE`), export GCS (`load_table_from_uri`). Comparamos tiempos y costes.

## Conexión con sesiones anteriores

| Sesión | Aporta | Uso aquí |
|---|---|---|
| S1+S2 | `silver_personio.dim_employee` (209 emp.) | Tabla histórica a respaldar |
| S2    | `gold_people_analytics.headcount_monthly` | Tabla derivada a respaldar |
| S3    | `predictions_retention.retention_actions` + 9 snapshots existentes | Auditamos sus snapshots y añadimos política |
| S3    | `pipeline_runs.retention_pipeline_runs` | Patrón para `pipeline_runs.restore_tests` |
| S4    | Código Dataform `dim_employee_scd2`, assertions | Se referencia, no se ejecuta aquí |

> **Importante**: el notebook **NO te enseña qué es Git**. Eso lo explica el instructor en clase con sus slides. El notebook asume que `git`, `gh` y `gcloud` están instalados y autenticados.

---
## Bloque 0 · Setup

Patrón idéntico al de M3/M4/M5/M6: detección Workbench/local, mismo proyecto, misma región.

In [1]:
# Dependencias (descomentar en primera ejecución)
# !pip install google-cloud-bigquery google-cloud-storage python-dotenv pyyaml

import os, sys, json, time, subprocess, tempfile, shutil
from pathlib import Path
from datetime import datetime, timezone, timedelta
import warnings
warnings.filterwarnings("ignore")

from google.cloud import bigquery, storage
from google.api_core.exceptions import NotFound, Conflict

# --- Detección de entorno (mismo patrón que M3-M6) ---
IN_VERTEX_AI = any([
    os.environ.get("DL_ANACONDA_HOME"),
    os.path.exists("/opt/deeplearning/metadata"),
])
if IN_VERTEX_AI:
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT")
    bq = bigquery.Client(project=PROJECT_ID, location="europe-southwest1")
    gcs = storage.Client(project=PROJECT_ID)
    print(f"Entorno: Vertex AI Workbench (ADC)")
else:
    from dotenv import load_dotenv
    from google.oauth2 import service_account
    load_dotenv(dotenv_path=Path("../.env"))
    PROJECT_ID = os.environ.get("GCP_PROJECT_ID")
    creds_path = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS", "../service-account.json")
    if creds_path and os.path.exists(creds_path):
        creds = service_account.Credentials.from_service_account_file(creds_path)
        bq = bigquery.Client(project=PROJECT_ID, credentials=creds, location="europe-southwest1")
        gcs = storage.Client(project=PROJECT_ID, credentials=creds)
    else:
        bq = bigquery.Client(project=PROJECT_ID, location="europe-southwest1")
        gcs = storage.Client(project=PROJECT_ID)
    print(f"Entorno: Local")

# Variables compartidas
REGION  = "europe-southwest1"
BUCKET  = os.environ.get("GCS_BUCKET_NAME", f"{PROJECT_ID}-datalake")
USER    = os.environ.get("USER", "carlos").replace(".", "_").lower()
SCRATCH = f"scratch_{USER}"
AUDIT   = "audit_archive"
TESTS   = "pipeline_runs"   # ya existe desde S3
REPO    = Path.cwd() / "repo_people_analytics_ops"  # directorio LOCAL del repo

# Tablas heredadas de sesiones previas (las que vamos a respaldar)
T_DIM_EMP        = f"{PROJECT_ID}.silver_personio.dim_employee"
T_RETENTION_ACT  = f"{PROJECT_ID}.predictions_retention.retention_actions"
T_HEADCOUNT      = f"{PROJECT_ID}.gold_people_analytics.headcount_monthly"

print(f"Proyecto:   {PROJECT_ID}")
print(f"Región:     {REGION}")
print(f"Bucket:     gs://{BUCKET}")
print(f"Datasets:   {SCRATCH} (scratch), {AUDIT} (snapshots), {TESTS} (logs)")
print(f"Repo local: {REPO}")

Entorno: Local
Proyecto:   project-9176af0b-ecb3-4050-859
Región:     europe-southwest1
Bucket:     gs://project-9176af0b-ecb3-4050-859-datalake
Datasets:   scratch_main (scratch), audit_archive (snapshots), pipeline_runs (logs)
Repo local: /Users/main/Desktop/curso-vertexai/sesion_05_actividad/repo_people_analytics_ops


In [2]:
# Habilitar APIs necesarias (idempotente). Si no tienes IAM no rompe.
APIS = ["bigquery.googleapis.com", "storage.googleapis.com",
        "secretmanager.googleapis.com", "cloudscheduler.googleapis.com"]
for api in APIS:
    r = subprocess.run(
        ["gcloud", "services", "enable", api, f"--project={PROJECT_ID}"],
        capture_output=True, text=True, timeout=60,
    )
    status = "OK" if r.returncode == 0 else "skip"
    print(f"  {api:.<45} {status}")

  bigquery.googleapis.com...................... OK
  storage.googleapis.com....................... OK
  secretmanager.googleapis.com................. OK
  cloudscheduler.googleapis.com................ OK


In [3]:
# Datasets que vamos a usar
def get_or_create_dataset(dataset_id, description, labels,
                          location=REGION, default_ttl_ms=None):
    full = f"{PROJECT_ID}.{dataset_id}"
    try:
        ds = bq.get_dataset(full)
        ds.description = description
        ds.labels = labels
        bq.update_dataset(ds, ["description", "labels"])
        print(f"  ya existe: {dataset_id}")
    except NotFound:
        ds = bigquery.Dataset(full)
        ds.location = location
        ds.description = description
        ds.labels = labels
        if default_ttl_ms:
            ds.default_table_expiration_ms = default_ttl_ms
        ds = bq.create_dataset(ds)
        print(f"  creado:    {dataset_id}")
    return ds

get_or_create_dataset(
    SCRATCH,
    description="Sandbox personal de S5. TTL 7 días. NO usar en prod.",
    labels={"team": "people-analytics", "env": "sandbox", "session": "s05"},
    default_ttl_ms=7 * 24 * 3600 * 1000,
)
get_or_create_dataset(
    AUDIT,
    description=("Snapshots inmutables para auditoría DPO. "
                 "Retención: 7 años. Acceso: solo DPO + DE-lead."),
    labels={"team": "people-analytics", "env": "prod", "contains_pii": "true"},
)
# pipeline_runs ya existe desde S3 — solo aseguramos labels
get_or_create_dataset(
    TESTS,
    description="Logs de ejecuciones de pipelines y restore tests.",
    labels={"team": "people-analytics", "env": "prod", "purpose": "observability"},
)

  creado:    scratch_main
  ya existe: audit_archive
  ya existe: pipeline_runs


Dataset(DatasetReference('project-9176af0b-ecb3-4050-859', 'pipeline_runs'))

---
## Bloque 1 · Estructura del repo

Creamos un directorio local `repo_people_analytics_ops/` que es **exactamente** lo que subirías a GitHub.

> Si tienes `gh` CLI autenticada y quieres subirlo de verdad, descomenta la celda de `gh repo create` al final del bloque.

**Estructura objetivo**:
```
repo_people_analytics_ops/
├── .github/
│   ├── workflows/
│   │   ├── validate-backups.yml
│   │   └── scheduled-snapshot.yml
│   ├── CODEOWNERS
│   └── pull_request_template.md
├── backups/
│   ├── sql/
│   ├── python/
│   ├── policies/
│   │   └── RETENTION.md
│   └── scripts/
│       └── check_retention_policy.py
└── README.md
```

In [4]:
# Crear el árbol vacío
DIRS = [
    REPO,
    REPO / ".github" / "workflows",
    REPO / "backups" / "sql",
    REPO / "backups" / "python",
    REPO / "backups" / "policies",
    REPO / "backups" / "scripts",
]
for d in DIRS:
    d.mkdir(parents=True, exist_ok=True)

# Inicializar git si no existe
if not (REPO / ".git").exists():
    subprocess.run(["git", "init", "-q", "-b", "main"], cwd=REPO, check=True)
    subprocess.run(["git", "config", "user.email", "carlos@imagina.com"], cwd=REPO, check=True)
    subprocess.run(["git", "config", "user.name", "Carlos"], cwd=REPO, check=True)
    print(f"git init en {REPO}")
else:
    print(f"git repo ya inicializado en {REPO}")

# Listar lo que hay
for p in sorted(REPO.rglob("*")):
    if ".git/" not in str(p):
        rel = p.relative_to(REPO)
        marker = "/" if p.is_dir() else ""
        print(f"  {rel}{marker}")

git init en /Users/main/Desktop/curso-vertexai/sesion_05_actividad/repo_people_analytics_ops
  .git/
  .github/
  .github/workflows/
  backups/
  backups/policies/
  backups/python/
  backups/scripts/
  backups/sql/


---
## Bloque 2 · Scripts de backup como código

Cada fichero generado en este bloque **es un artefacto versionable**. El alumno SIENTE que:
- M10 (lo que hace) y M9 (cómo se gobierna el cambio) son la misma pieza.
- El snapshot de `retention_actions` que produce el cron diario **es el mismo SQL** que el reviewer aprobó en un PR.

In [5]:
# 2.1 — SQL: snapshot diario de retention_actions (la tabla que S3 produjo)
sql_snap_retention = """-- backups/sql/snapshot_retention_actions.sql
-- Snapshot diario de predictions_retention.retention_actions
-- Frecuencia: diaria 03:15 UTC (scheduled-snapshot.yml)
-- Retención: 7 años (expiration_timestamp abajo)
-- Owner: data-engineering@imagina.com
-- CODEOWNERS: cualquier cambio aquí requiere review de @dpo

DECLARE snap_name STRING;
SET snap_name = FORMAT('retention_actions_snap_%t', CURRENT_DATE());

EXECUTE IMMEDIATE FORMAT(\"\"\"
  CREATE SNAPSHOT TABLE `__PROJECT__.audit_archive.%s`
  CLONE `__PROJECT__.predictions_retention.retention_actions`
  OPTIONS (
    expiration_timestamp = TIMESTAMP_ADD(CURRENT_TIMESTAMP(), INTERVAL 2555 DAY),
    description = 'Snapshot diario para auditoría DPO. Retención 7 años.',
    labels = [('purpose','audit'),('source','retention_actions'),('contains_pii','false')]
  )
\"\"\", snap_name);
""".replace("__PROJECT__", PROJECT_ID)

(REPO / "backups" / "sql" / "snapshot_retention_actions.sql").write_text(sql_snap_retention)
print("escrito backups/sql/snapshot_retention_actions.sql")
print("\n--- preview ---")
print(sql_snap_retention[:600])

escrito backups/sql/snapshot_retention_actions.sql

--- preview ---
-- backups/sql/snapshot_retention_actions.sql
-- Snapshot diario de predictions_retention.retention_actions
-- Frecuencia: diaria 03:15 UTC (scheduled-snapshot.yml)
-- Retención: 7 años (expiration_timestamp abajo)
-- Owner: data-engineering@imagina.com
-- CODEOWNERS: cualquier cambio aquí requiere review de @dpo

DECLARE snap_name STRING;
SET snap_name = FORMAT('retention_actions_snap_%t', CURRENT_DATE());

EXECUTE IMMEDIATE FORMAT("""
  CREATE SNAPSHOT TABLE `project-9176af0b-ecb3-4050-859.audit_archive.%s`
  CLONE `project-9176af0b-ecb3-4050-859.predictions_retention.retention_actions`
  OP


In [6]:
# 2.2 — SQL: snapshot semanal de dim_employee (tabla histórica con PII real)
sql_snap_dim_emp = """-- backups/sql/snapshot_dim_employee.sql
-- Snapshot semanal de silver_personio.dim_employee
-- Tabla CON PII real (209 empleados, gross_salary_annual, IBAN-equivalentes)
-- Frecuencia: semanal (domingos 04:00 UTC)
-- Retención: 7 años (compliance laboral)
-- CODEOWNERS: cualquier cambio requiere review de @dpo Y @data-eng-lead

DECLARE snap_name STRING;
SET snap_name = FORMAT('dim_employee_snap_%t', CURRENT_DATE());

EXECUTE IMMEDIATE FORMAT(\"\"\"
  CREATE SNAPSHOT TABLE `__PROJECT__.audit_archive.%s`
  CLONE `__PROJECT__.silver_personio.dim_employee`
  OPTIONS (
    expiration_timestamp = TIMESTAMP_ADD(CURRENT_TIMESTAMP(), INTERVAL 2555 DAY),
    description = 'Snapshot semanal dim_employee. PII real. Retención 7 años (Art. 5(1)(e) GDPR).',
    labels = [('purpose','audit'),('source','dim_employee'),('contains_pii','true')]
  )
\"\"\", snap_name);
""".replace("__PROJECT__", PROJECT_ID)

(REPO / "backups" / "sql" / "snapshot_dim_employee.sql").write_text(sql_snap_dim_emp)
print("escrito backups/sql/snapshot_dim_employee.sql")

escrito backups/sql/snapshot_dim_employee.sql


In [7]:
# 2.3 — SQL: políticas de retención (partition_expiration + max_time_travel)
sql_retention = """-- backups/sql/retention_policies.sql
-- Política de retención aplicada por código (NO por consola)
-- Mantenida en sync con backups/policies/RETENTION.md
-- Owner: @data-eng-lead | Aprobador: @dpo

-- 1. fact_salary_history particionada → 7 años (compliance laboral)
ALTER TABLE `__PROJECT__.silver_personio.fact_salary_history`
SET OPTIONS (
  partition_expiration_days = 2555,  -- 7 años
  description = 'Histórico salarial mensual. Retención 7 años (GDPR + laboral).'
);

-- 2. fact_payroll_monthly → 7 años (mismo motivo)
ALTER TABLE `__PROJECT__.silver_personio.fact_payroll_monthly`
SET OPTIONS (
  partition_expiration_days = 2555,
  description = 'Nómina mensual. Retención 7 años.'
);

-- 3. max_time_travel_hours = 168 (7 días, el máximo) en datasets con PII
ALTER SCHEMA `__PROJECT__.silver_personio`
SET OPTIONS (max_time_travel_hours = 168);

ALTER SCHEMA `__PROJECT__.predictions_retention`
SET OPTIONS (max_time_travel_hours = 168);

-- 4. Sandbox: time travel mínimo (no es producción, no merece coste)
ALTER SCHEMA `__PROJECT__.audit_archive`
SET OPTIONS (max_time_travel_hours = 168);
""".replace("__PROJECT__", PROJECT_ID)

(REPO / "backups" / "sql" / "retention_policies.sql").write_text(sql_retention)
print("escrito backups/sql/retention_policies.sql")

escrito backups/sql/retention_policies.sql


In [8]:
# 2.4 — Python: export semanal a GCS Parquet (última línea de defensa)
py_export = """\"\"\"
backups/python/export_to_gcs.py
Exporta tablas críticas a GCS Parquet+Snappy.
Frecuencia: semanal (domingos 04:30 UTC).
Lifecycle policy del bucket mueve a Coldline a 30d, Archive a 90d, Delete a 7y.
\"\"\"
import os
from datetime import datetime, timezone
from google.cloud import bigquery

PROJECT  = os.environ['GCP_PROJECT_ID']
BUCKET   = os.environ['GCS_BUCKET_NAME']
LOCATION = os.environ.get('GCP_REGION', 'europe-southwest1')

CRITICAL = [
    'predictions_retention.retention_actions',
    'silver_personio.dim_employee',
    'silver_personio.fact_salary_history',
    'gold_people_analytics.headcount_monthly',
]

def export_table(client, fqn, today):
    ds, tbl = fqn.split('.', 1)
    uri = f'gs://{BUCKET}/backups/{ds}/{tbl}/{today}/data-*.parquet'
    job = client.extract_table(
        f'{PROJECT}.{fqn}', uri,
        job_config=bigquery.ExtractJobConfig(
            destination_format=bigquery.DestinationFormat.PARQUET,
            compression='SNAPPY',
        ),
        location=LOCATION,
    )
    job.result()
    return uri

def main():
    client = bigquery.Client(project=PROJECT, location=LOCATION)
    today = datetime.now(timezone.utc).strftime('%Y-%m-%d')
    for fqn in CRITICAL:
        uri = export_table(client, fqn, today)
        print(f'OK  {fqn:60s} -> {uri}')

if __name__ == '__main__':
    main()
"""
(REPO / "backups" / "python" / "export_to_gcs.py").write_text(py_export)
print("escrito backups/python/export_to_gcs.py")

escrito backups/python/export_to_gcs.py


In [9]:
# 2.5 — Python: restore-test mensual (la prueba que evita auditorías humillantes)
py_restore = """\"\"\"
backups/python/restore_test.py
Job mensual: prueba que el último snapshot de retention_actions es restaurable.
- Localiza el último snapshot
- Lo clona a scratch
- Compara checksum (filas) con producción
- Reporta resultado a pipeline_runs.restore_tests
- Limpia el clone
\"\"\"
import os
from datetime import datetime, timezone
from google.cloud import bigquery

PROJECT  = os.environ['GCP_PROJECT_ID']
LOCATION = os.environ.get('GCP_REGION', 'europe-southwest1')

SOURCE   = f'{PROJECT}.predictions_retention.retention_actions'
SCRATCH  = f'{PROJECT}.scratch_restoretest.retention_actions_test'
LOGS     = f'{PROJECT}.pipeline_runs.restore_tests'

def ensure_log_table(client):
    schema = [
        bigquery.SchemaField('run_id', 'STRING'),
        bigquery.SchemaField('snapshot_used', 'STRING'),
        bigquery.SchemaField('ok', 'BOOL'),
        bigquery.SchemaField('src_rows', 'INT64'),
        bigquery.SchemaField('snap_rows', 'INT64'),
        bigquery.SchemaField('ts', 'TIMESTAMP'),
    ]
    table = bigquery.Table(LOGS, schema=schema)
    try:
        client.create_table(table)
    except Exception:
        pass

def main():
    client = bigquery.Client(project=PROJECT, location=LOCATION)
    ensure_log_table(client)
    run_id = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')

    snaps = list(client.query(f\"\"\"
        SELECT table_name
        FROM `{PROJECT}.audit_archive`.INFORMATION_SCHEMA.TABLE_SNAPSHOTS
        WHERE base_table_name = 'retention_actions'
        ORDER BY snapshot_time DESC
        LIMIT 1
    \"\"\").result())
    if not snaps:
        raise RuntimeError('No hay snapshots disponibles para restore_test')
    latest = snaps[0].table_name
    snap_fqn = f'{PROJECT}.audit_archive.{latest}'

    # Asegurar scratch dataset
    try:
        client.create_dataset(f'{PROJECT}.scratch_restoretest', exists_ok=True)
    except Exception:
        pass
    client.query(f'DROP TABLE IF EXISTS `{SCRATCH}`').result()
    client.query(f'CREATE TABLE `{SCRATCH}` CLONE `{snap_fqn}`').result()

    def count(t):
        return list(client.query(f'SELECT COUNT(*) c FROM `{t}`').result())[0].c
    src_c, snap_c = count(SOURCE), count(SCRATCH)
    ok = snap_c >= 1  # criterio mínimo: el snapshot contiene datos

    client.query(f\"\"\"
        INSERT INTO `{LOGS}` (run_id, snapshot_used, ok, src_rows, snap_rows, ts)
        VALUES ('{run_id}', '{latest}', {ok}, {src_c}, {snap_c}, CURRENT_TIMESTAMP())
    \"\"\").result()

    client.delete_table(SCRATCH, not_found_ok=True)
    print(f'restore_test {run_id}: {\"OK\" if ok else \"FAIL\"} (src={src_c}, snap={snap_c})')

if __name__ == '__main__':
    main()
"""
(REPO / "backups" / "python" / "restore_test.py").write_text(py_restore)
print("escrito backups/python/restore_test.py")

escrito backups/python/restore_test.py


In [10]:
# 2.6 — Política de retención en Markdown (lo que firma el DPO)
retention_md = f"""# Política de retención de datos — People Analytics

**Owner**: Data Engineering Lead
**Aprobador**: DPO (Data Protection Officer)
**Última revisión**: {datetime.now(timezone.utc).strftime('%Y-%m-%d')}
**Versionado**: este fichero. Cambios requieren PR con review de `@dpo`.

---

## 1. Marco legal

- **GDPR Art. 5(1)(e)** — Limitación del plazo de conservación
- **GDPR Art. 32** — Seguridad del tratamiento (backups, recuperación)
- **LOPDGDD Art. 32** — Bloqueo de datos
- **Estatuto de los Trabajadores Art. 4.2.e** — Conservación de información laboral

## 2. Categorías de datos y plazos

| Categoría | Tabla(s) | Plazo retención | Justificación |
|---|---|---|---|
| Bronze raw | `bronze_personio.*` | 90 días | Recarga sin retrabajo |
| Silver histórico | `silver_personio.fact_*` | 7 años (2555 días) | Compliance laboral |
| Silver maestro | `silver_personio.dim_employee` | 7 años | Auditoría |
| Gold derivados | `gold_people_analytics.*` | Indefinida (reconstruibles) | Métricas oficiales |
| Predicciones | `predictions_retention.*` | 2 años | Trazabilidad modelo (XAI) |
| Logs pipeline | `pipeline_runs.*` | 1 año | Operativa SRE |
| Sandbox | `scratch_*` | 7 días | Experimentación |

## 3. Métodos de respaldo (defensa en profundidad)

| Capa | Mecanismo | Ventana | Coste relativo |
|---|---|---|---|
| 1 | Time travel BigQuery | 0-7 días | Incluido (physical billing factura) |
| 2 | Snapshot diario en `audit_archive` | 7 años | Bajo (CoW con physical billing) |
| 3 | Table clone bajo demanda | N/A | Bajo (sólo experimentación) |
| 4 | Export semanal a GCS Parquet | 7 años | Mínimo (Archive: $0.0012/GB·mes) |

## 4. Pruebas de restauración

- **Frecuencia**: mensual (1er lunes del mes)
- **Responsable**: SRE de guardia
- **Job**: `backups/python/restore_test.py`
- **Registro**: `pipeline_runs.restore_tests`
- **Criterio de éxito**: el snapshot del mes anterior contiene ≥ 1 fila

## 5. RTO/RPO por categoría

| Tipo | RTO | RPO | Estrategia primaria |
|---|---|---|---|
| Operativo crítico (`silver_personio`) | 1h | 1h | Time travel + snapshot diario |
| Derivado (`gold_*`) | 4h | 24h | Snapshot diario |
| Modelo (`predictions_*`) | 4h | 24h | Snapshot diario + retrain on demand |
| Logs (`pipeline_runs`) | 24h | 24h | Snapshot semanal |

## 6. Separación de entornos

- `proyecto-prod` ↔ `proyecto-dev` en proyectos GCP **distintos**
- Service Accounts de dev **NO** tienen IAM en prod
- Datos de prod **NO** cruzan a dev sin paso por Cloud DLP (Sesión 6)

## 7. Aprobaciones

- DPO: ____________________ (firma vía PR review)
- CISO: ____________________ (firma vía PR review)
"""
(REPO / "backups" / "policies" / "RETENTION.md").write_text(retention_md)
print("escrito backups/policies/RETENTION.md")
print(f"  ({len(retention_md.splitlines())} líneas)")

escrito backups/policies/RETENTION.md
  (64 líneas)


In [11]:
# 2.7 — Script CI: check_retention_policy.py
# Convierte la POLÍTICA escrita en una REGLA EJECUTABLE.
# Si un PR baja partition_expiration_days a <90 en una tabla con PII, el CI falla.
check_py = """\"\"\"
backups/scripts/check_retention_policy.py
Chequeo CI: ningún SQL de backups/sql/ debe poner partition_expiration_days < 90
en una tabla cuyo nombre sugiera PII.
\"\"\"
import re, sys, glob
from pathlib import Path

PII_HINTS = ('payroll', 'salary', 'dim_employee', 'history', 'gross', 'iban', 'retention_actions')
THRESHOLD = 90

def check_file(path: Path):
    errs = []
    text = path.read_text()
    pat = re.compile(r'partition_expiration_days\\s*=\\s*(\\d+)', re.IGNORECASE)
    for m in pat.finditer(text):
        days = int(m.group(1))
        if days < THRESHOLD and any(h in text.lower() for h in PII_HINTS):
            errs.append(f'{path}: partition_expiration_days={days} (<{THRESHOLD}) con hint PII')
    return errs

def main():
    errs = []
    for p in glob.glob('backups/sql/**/*.sql', recursive=True):
        errs.extend(check_file(Path(p)))
    if errs:
        print('POLICY VIOLATIONS:')
        for e in errs: print(' -', e)
        sys.exit(1)
    print('OK: retention policy compliant')

if __name__ == '__main__':
    main()
"""
(REPO / "backups" / "scripts" / "check_retention_policy.py").write_text(check_py)
print("escrito backups/scripts/check_retention_policy.py")

# Validar que compila
import py_compile
py_compile.compile(str(REPO / "backups" / "scripts" / "check_retention_policy.py"), doraise=True)
print("OK compila")

escrito backups/scripts/check_retention_policy.py
OK compila


In [12]:
# 2.8 — Probar el script CI en local (debe pasar)
result = subprocess.run(
    [sys.executable, "backups/scripts/check_retention_policy.py"],
    cwd=REPO, capture_output=True, text=True,
)
print("STDOUT:", result.stdout)
print("STDERR:", result.stderr)
print("RC:", result.returncode)
assert result.returncode == 0, "El check de política debe pasar sobre los SQL que escribimos"
print("\n✓ Política CI verde sobre los SQL del repo")

STDOUT: OK: retention policy compliant

STDERR: 
RC: 0

✓ Política CI verde sobre los SQL del repo


---
## Bloque 3 · CI/CD desde el notebook

Generamos los dos workflows que se ejecutarían en GitHub Actions:

1. **`validate-backups.yml`** — corre en cada PR. Lint SQL + chequeo de política. Falla rojo si alguien baja retention de PII sin pasar por revisión.
2. **`scheduled-snapshot.yml`** — corre cada noche a las 03:15 UTC. Ejecuta los SQL de `backups/sql/snapshot_*.sql`.

> Estos `.yml` se VALIDAN como YAML aquí mismo. No los ejecutamos contra GitHub Actions desde el notebook (no tendría sentido). Si subes el repo a GitHub, se activan solos al primer push.

In [13]:
import yaml

# 3.1 — validate-backups.yml
validate_yml = """
name: validate-backups
on:
  pull_request:
    paths:
      - 'backups/**'
      - '.github/workflows/**'
  workflow_dispatch:

permissions:
  contents: read

jobs:
  lint-sql:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: '3.11'
      - run: pip install sqlfluff==3.0.7
      - name: Lint SQL
        run: sqlfluff lint backups/sql --dialect bigquery --disable-progress-bar

  policy-check:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: '3.11'
      - name: Retention policy compliance
        run: python backups/scripts/check_retention_policy.py
"""
parsed = yaml.safe_load(validate_yml)
assert "jobs" in parsed and len(parsed["jobs"]) == 2
(REPO / ".github" / "workflows" / "validate-backups.yml").write_text(validate_yml.lstrip())
print(f"OK validate-backups.yml ({len(parsed['jobs'])} jobs)")

OK validate-backups.yml (2 jobs)


In [14]:
# 3.2 — scheduled-snapshot.yml (cron diario)
sched_yml = """
name: scheduled-snapshot
on:
  schedule:
    - cron: '15 3 * * *'   # 03:15 UTC daily
  workflow_dispatch:

permissions:
  contents: read
  id-token: write   # para Workload Identity Federation

jobs:
  snapshot:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - id: auth
        uses: google-github-actions/auth@v2
        with:
          workload_identity_provider: ${{ secrets.WIF_PROVIDER }}
          service_account: ${{ secrets.CI_SA }}
      - uses: google-github-actions/setup-gcloud@v2
      - name: Ejecutar snapshots diarios
        run: |
          for f in backups/sql/snapshot_*.sql; do
            echo "Running $f"
            bq query --use_legacy_sql=false --location=europe-southwest1 < "$f"
          done
"""
parsed = yaml.safe_load(sched_yml)
assert "jobs" in parsed and "snapshot" in parsed["jobs"]
(REPO / ".github" / "workflows" / "scheduled-snapshot.yml").write_text(sched_yml.lstrip())
print(f"OK scheduled-snapshot.yml ({len(parsed['jobs'])} jobs, cron diario)")

OK scheduled-snapshot.yml (1 jobs, cron diario)


In [15]:
# 3.3 — CODEOWNERS: quién firma qué
codeowners = """# CODEOWNERS — políticas y respaldos
# Sintaxis: <path> <owner1> [<owner2>]
# Cambios en estos paths requieren approval del owner correspondiente.

# Cualquier cambio en política de retención exige firma del DPO
/backups/policies/   @dpo @data-eng-lead

# SQL de backups: DE-lead + senior DE
/backups/sql/        @data-eng-lead @senior-de

# Scripts Python: solo DE
/backups/python/     @senior-de @data-eng-lead

# Workflows CI/CD: DE-lead + DevOps
/.github/            @data-eng-lead @devops
"""
(REPO / ".github" / "CODEOWNERS").write_text(codeowners)
print("OK CODEOWNERS")

OK CODEOWNERS


In [16]:
# 3.4 — Template de PR (checklist GDPR-aware)
pr_template = """## Qué cambia

…

## Por qué

…

## Impacto en datos

- Tablas afectadas:
- Volumen estimado:
- Retención implicada:

## Checklist GDPR

- [ ] No introduzco columnas PII en `gold_*` sin Policy Tag
- [ ] Mantengo `partition_expiration_days ≥ 90` para tablas con PII
- [ ] He probado el SQL en local con `bq query --dry_run`
- [ ] He añadido / actualizado test/assertion donde aplica
- [ ] Si toco política de retención, he añadido al DPO al PR

## Cómo probarlo

```
…
```

## Issue relacionado

Closes #…
"""
(REPO / ".github" / "pull_request_template.md").write_text(pr_template)
print("OK pull_request_template.md")

OK pull_request_template.md


In [17]:
# 3.5 — README del repo (la "carta de presentación")
readme = f"""# people-analytics-ops

Repo de operativa para el data warehouse de People Analytics en GCP.

## Estructura

- `backups/sql/` — SQL de snapshots y políticas de retención (un fichero = un job)
- `backups/python/` — exports a GCS + restore-test mensual
- `backups/policies/RETENTION.md` — política firmada por DPO
- `backups/scripts/` — chequeos CI (policy enforcement)
- `.github/workflows/` — CI (`validate-backups`) y cron (`scheduled-snapshot`)

## Branching

- `main` → producción (release tags `vX.Y.Z`)
- `develop` → integración
- `feature/*` → trabajo en curso

`main` está protegida: PR + 1 review + CODEOWNERS + CI verde.

## Setup local

```
gcloud auth application-default login
export GCP_PROJECT_ID={PROJECT_ID}
export GCS_BUCKET_NAME={BUCKET}
export GCP_REGION=europe-southwest1
```

## Cómo lanzar manualmente un snapshot

```
bq query --use_legacy_sql=false --location=europe-southwest1 \\
    < backups/sql/snapshot_retention_actions.sql
```

## Cómo probar el restore

```
python backups/python/restore_test.py
```
"""
(REPO / "README.md").write_text(readme)
print("OK README.md")

OK README.md


In [18]:
# 3.6 — Primer commit real (en local; opcional push a GitHub al final)
subprocess.run(["git", "add", "."], cwd=REPO, check=True)
status = subprocess.run(["git", "status", "--short"], cwd=REPO,
                        capture_output=True, text=True).stdout
print("Archivos en stage:")
print(status)

commit = subprocess.run(
    ["git", "commit", "-m", "feat(backups): initial backup-as-code scaffolding"],
    cwd=REPO, capture_output=True, text=True
)
print("STDOUT:", commit.stdout)
print("STDERR:", commit.stderr)
# Mostrar log
log = subprocess.run(["git", "log", "--oneline"], cwd=REPO,
                     capture_output=True, text=True).stdout
print("\nGit log:")
print(log)

Archivos en stage:
A  .github/CODEOWNERS
A  .github/pull_request_template.md
A  .github/workflows/scheduled-snapshot.yml
A  .github/workflows/validate-backups.yml
A  README.md
A  backups/policies/RETENTION.md
A  backups/python/export_to_gcs.py
A  backups/python/restore_test.py
A  backups/scripts/__pycache__/check_retention_policy.cpython-313.pyc
A  backups/scripts/check_retention_policy.py
A  backups/sql/retention_policies.sql
A  backups/sql/snapshot_dim_employee.sql
A  backups/sql/snapshot_retention_actions.sql

STDOUT: [main (root-commit) 0ce021d] feat(backups): initial backup-as-code scaffolding
 13 files changed, 429 insertions(+)
 create mode 100644 .github/CODEOWNERS
 create mode 100644 .github/pull_request_template.md
 create mode 100644 .github/workflows/scheduled-snapshot.yml
 create mode 100644 .github/workflows/validate-backups.yml
 create mode 100644 README.md
 create mode 100644 backups/policies/RETENTION.md
 create mode 100644 backups/python/export_to_gcs.py
 create mode 

---
## Bloque 4 · Catástrofe + 3 caminos de recuperación

**Escenario**: creamos `scratch_<usuario>.demo_critical` con un subset real de `dim_employee`. La borramos (`DROP TABLE`). La recuperamos por tres caminos diferentes. Comparamos.

> Esto es **el día que importa**. La pregunta "¿funciona mi backup?" se contesta haciéndolo, no leyéndolo.

In [19]:
# 4.0 — Crear la tabla "crítica" con datos reales de S1+S2
DEMO_TABLE = f"{PROJECT_ID}.{SCRATCH}.demo_critical"

bq.query(f"""
  CREATE OR REPLACE TABLE `{DEMO_TABLE}` AS
  SELECT
    employee_code, first_name, last_name, country, band,
    gross_salary_annual, hire_date, status
  FROM `{T_DIM_EMP}`
  LIMIT 100
""").result()

# Verificar
n = list(bq.query(f"SELECT COUNT(*) c FROM `{DEMO_TABLE}`").result())[0].c
print(f"creada {DEMO_TABLE}: {n} filas")

creada project-9176af0b-ecb3-4050-859.scratch_main.demo_critical: 100 filas


In [20]:
# 4.1 — Crear un snapshot ANTES de la catástrofe (lo que haríamos en producción)
SNAP_NAME = f"demo_critical_snap_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
SNAP_FQN  = f"{PROJECT_ID}.{AUDIT}.{SNAP_NAME}"

bq.query(f"""
  CREATE SNAPSHOT TABLE `{SNAP_FQN}`
  CLONE `{DEMO_TABLE}`
  OPTIONS (
    expiration_timestamp = TIMESTAMP_ADD(CURRENT_TIMESTAMP(), INTERVAL 1 DAY),
    description = "Snapshot demo S5 — auto-expira en 24h"
  )
""").result()
print(f"snapshot creado: {SNAP_FQN}")

# Verificar que aparece en INFORMATION_SCHEMA.TABLE_SNAPSHOTS
rows = list(bq.query(f"""
  SELECT table_name, snapshot_time, base_table_name
  FROM `{PROJECT_ID}.{AUDIT}`.INFORMATION_SCHEMA.TABLE_SNAPSHOTS
  WHERE table_name = '{SNAP_NAME}'
""").result())
for r in rows:
    print(f"  {r.table_name} from {r.base_table_name} @ {r.snapshot_time}")

snapshot creado: project-9176af0b-ecb3-4050-859.audit_archive.demo_critical_snap_20260514_144845
  demo_critical_snap_20260514_144845 from demo_critical @ 2026-05-14 14:48:46.052000+00:00


In [21]:
# 4.2 — Export a GCS Parquet (la última línea)
export_uri = f"gs://{BUCKET}/backups/scratch/demo_critical/data-*.parquet"

job = bq.extract_table(
    DEMO_TABLE,
    export_uri,
    job_config=bigquery.ExtractJobConfig(
        destination_format=bigquery.DestinationFormat.PARQUET,
        compression="SNAPPY",
    ),
    location=REGION,
)
job.result()

# Listar lo que escribió
b = gcs.get_bucket(BUCKET)
files = list(b.list_blobs(prefix="backups/scratch/demo_critical/"))
total_bytes = sum(f.size for f in files)
print(f"exportados {len(files)} fichero(s), total {total_bytes:,} bytes")
for f in files:
    print(f"  gs://{BUCKET}/{f.name} ({f.size} bytes)")

exportados 1 fichero(s), total 5,163 bytes
  gs://project-9176af0b-ecb3-4050-859-datalake/backups/scratch/demo_critical/data-000000000000.parquet (5163 bytes)


In [22]:
# 4.3 — Capturar timestamp justo ANTES del DROP (lo necesitaremos para time travel)
ts_before_drop = list(bq.query("SELECT UNIX_MILLIS(CURRENT_TIMESTAMP()) AS t").result())[0].t
print(f"timestamp pre-drop (ms): {ts_before_drop}")

# Dar 2 segundos de margen porque @ms necesita un punto en el que la tabla existió
time.sleep(2)

# 💥 LA CATÁSTROFE
bq.query(f"DROP TABLE `{DEMO_TABLE}`").result()
print(f"💥 DROP TABLE ejecutado sobre {DEMO_TABLE}")

# Verificar que está muerta
try:
    bq.get_table(DEMO_TABLE)
    print("(la tabla sigue ahí — algo no funcionó)")
except NotFound:
    print("✓ la tabla ya no existe")

timestamp pre-drop (ms): 1778770131480
💥 DROP TABLE ejecutado sobre project-9176af0b-ecb3-4050-859.scratch_main.demo_critical
✓ la tabla ya no existe


### Recuperación A — Time travel (decorador `@ms`)

Cuando un `DROP TABLE` acaba de ocurrir, **no puedes** usar `FOR SYSTEM_TIME AS OF` por nombre porque la tabla ya no se resuelve. La vía documentada es **`bq cp tabla@<ms>`** (Python: `client.copy_table(f"...@{ms}", dst)`).

Ventana: 7 días. Si pasó más, no funciona.

In [23]:
import time as _t
# 4.4 — Recuperación A: time travel vía copy_table con @ms decorator
t0 = _t.time()
src_decorated = f"{PROJECT_ID}.{SCRATCH}.demo_critical@{ts_before_drop}"
dst_a = f"{PROJECT_ID}.{SCRATCH}.demo_critical_recovered_A_timetravel"

# Idempotencia: borrar destino si quedó de una ejecución previa
bq.delete_table(dst_a, not_found_ok=True)

job = bq.copy_table(src_decorated, dst_a, location=REGION)
job.result()
elapsed_a = _t.time() - t0

n = list(bq.query(f"SELECT COUNT(*) c FROM `{dst_a}`").result())[0].c
print(f"✓ recuperada vía time travel: {n} filas en {elapsed_a:.2f}s")

✓ recuperada vía time travel: 100 filas en 2.11s


### Recuperación B — Snapshot (`CLONE`)

Si pasó más de 7 días o si la tabla fue recreada (lo que invalida el time travel del original), usamos el snapshot que creamos antes del incidente.

Ventana: la que diga `expiration_timestamp` del snapshot (en producción, 7 años).

In [24]:
# 4.5 — Recuperación B: desde el snapshot
t0 = _t.time()
dst_b = f"{PROJECT_ID}.{SCRATCH}.demo_critical_recovered_B_snapshot"
bq.delete_table(dst_b, not_found_ok=True)
bq.query(f"""
  CREATE TABLE `{dst_b}`
  CLONE `{SNAP_FQN}`
""").result()
elapsed_b = _t.time() - t0

n = list(bq.query(f"SELECT COUNT(*) c FROM `{dst_b}`").result())[0].c
print(f"✓ recuperada vía snapshot: {n} filas en {elapsed_b:.2f}s")

✓ recuperada vía snapshot: 100 filas en 1.27s


### Recuperación C — Export GCS (última línea)

Si los snapshots se corrompieron, o llevas más de 7 años sin acceder al dato, o estás recuperando un proyecto entero, vas al Parquet en GCS. Más lento pero **el más durable**.

In [25]:
# 4.6 — Recuperación C: load desde GCS Parquet
t0 = _t.time()
dst_c = f"{PROJECT_ID}.{SCRATCH}.demo_critical_recovered_C_gcs"
bq.delete_table(dst_c, not_found_ok=True)

job = bq.load_table_from_uri(
    f"gs://{BUCKET}/backups/scratch/demo_critical/data-*.parquet",
    dst_c,
    job_config=bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.PARQUET,
        write_disposition="WRITE_TRUNCATE",
    ),
    location=REGION,
)
job.result()
elapsed_c = _t.time() - t0

n = list(bq.query(f"SELECT COUNT(*) c FROM `{dst_c}`").result())[0].c
print(f"✓ recuperada vía GCS Parquet: {n} filas en {elapsed_c:.2f}s")

✓ recuperada vía GCS Parquet: 100 filas en 3.74s


In [26]:
# 4.7 — Resumen comparativo de los 3 caminos
import pandas as pd
comparison = pd.DataFrame([
    {"método": "Time travel @ms",  "ventana": "7 días",     "tiempo (s)": round(elapsed_a, 2), "comando": "client.copy_table(f'tbl@{ms}', dst)"},
    {"método": "Snapshot CLONE",   "ventana": "configurable","tiempo (s)": round(elapsed_b, 2), "comando": "CREATE TABLE … CLONE snapshot"},
    {"método": "Load Parquet GCS", "ventana": "indefinida", "tiempo (s)": round(elapsed_c, 2), "comando": "client.load_table_from_uri(...)"},
])
display(comparison)

print("""
Lectura honesta del resultado:
- Time travel suele ser el más rápido, pero solo si has actuado en < 7 días.
- Snapshot es el sweet spot para auditoría: barato (CoW) y rápido.
- GCS es la única defensa real ante catástrofes que destruyen BQ entero.
- Los tres caminos DEBEN existir en producción. No son alternativas.
""")

,método,ventana,tiempo (s),comando
0,Time travel @ms,7 días,2.11,"client.copy_table(f'tbl@{ms}', dst)"
1,Snapshot CLONE,configurable,1.27,CREATE TABLE … CLONE snapshot
2,Load Parquet GCS,indefinida,3.74,client.load_table_from_uri(...)



Lectura honesta del resultado:
- Time travel suele ser el más rápido, pero solo si has actuado en < 7 días.
- Snapshot es el sweet spot para auditoría: barato (CoW) y rápido.
- GCS es la única defensa real ante catástrofes que destruyen BQ entero.
- Los tres caminos DEBEN existir en producción. No son alternativas.



---
## Bloque 5 · Restore-test mensual

Ejecutamos el `restore_test.py` que generamos en el bloque 2, **contra los snapshots reales de S3** que ya existen en `predictions_retention.*`.

Esto es la prueba de fuego: si tu plan de backup funciona, este script debe pasar.

In [27]:
# 5.1 — Asegurar tabla de log existe
schema = [
    bigquery.SchemaField('run_id', 'STRING'),
    bigquery.SchemaField('snapshot_used', 'STRING'),
    bigquery.SchemaField('ok', 'BOOL'),
    bigquery.SchemaField('src_rows', 'INT64'),
    bigquery.SchemaField('snap_rows', 'INT64'),
    bigquery.SchemaField('ts', 'TIMESTAMP'),
]
LOG_TABLE = f"{PROJECT_ID}.{TESTS}.restore_tests"
try:
    bq.create_table(bigquery.Table(LOG_TABLE, schema=schema))
    print(f"creada {LOG_TABLE}")
except Conflict:
    print(f"ya existe {LOG_TABLE}")

creada project-9176af0b-ecb3-4050-859.pipeline_runs.restore_tests


In [28]:
# 5.2 — Ejecutar el restore-test (versión inline para la clase)
run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

# Localizar el snapshot más reciente de retention_actions
snaps = list(bq.query(f"""
  SELECT table_name
  FROM `{PROJECT_ID}.predictions_retention`.INFORMATION_SCHEMA.TABLE_SNAPSHOTS
  WHERE base_table_name = 'retention_actions'
  ORDER BY snapshot_time DESC LIMIT 1
""").result())
assert snaps, "No hay snapshots de retention_actions — esperado tras S3"
latest_snap = snaps[0].table_name
snap_fqn = f"{PROJECT_ID}.predictions_retention.{latest_snap}"
print(f"último snapshot localizado: {latest_snap}")

# Clonar a scratch
TEST_DS = "scratch_restoretest"
try:
    bq.create_dataset(f"{PROJECT_ID}.{TEST_DS}", exists_ok=True)
except Exception:
    pass

test_tbl = f"{PROJECT_ID}.{TEST_DS}.retention_actions_test_{run_id}"
bq.query(f"CREATE TABLE `{test_tbl}` CLONE `{snap_fqn}`").result()

# Checksums
def count(t):
    return list(bq.query(f"SELECT COUNT(*) c FROM `{t}`").result())[0].c
src_c  = count(T_RETENTION_ACT)
snap_c = count(test_tbl)
ok = snap_c >= 1

# Log
bq.query(f"""
  INSERT INTO `{LOG_TABLE}` (run_id, snapshot_used, ok, src_rows, snap_rows, ts)
  VALUES ('{run_id}', '{latest_snap}', {ok}, {src_c}, {snap_c}, CURRENT_TIMESTAMP())
""").result()

# Cleanup
bq.delete_table(test_tbl, not_found_ok=True)

print(f"\nrestore_test {run_id}")
print(f"  snapshot usado: {latest_snap}")
print(f"  src_rows:       {src_c}")
print(f"  snap_rows:      {snap_c}")
print(f"  resultado:      {'✓ OK' if ok else '✗ FAIL'}")

último snapshot localizado: retention_actions_snapshot_20251101_20260508

restore_test 20260514_144903
  snapshot usado: retention_actions_snapshot_20251101_20260508
  src_rows:       316
  snap_rows:      316
  resultado:      ✓ OK


In [29]:
# 5.3 — Ver el histórico de restore_tests (de momento solo este)
rows = list(bq.query(f"""
  SELECT run_id, snapshot_used, ok, src_rows, snap_rows, ts
  FROM `{LOG_TABLE}`
  ORDER BY ts DESC LIMIT 10
""").result())

import pandas as pd
df = pd.DataFrame([dict(r) for r in rows])
display(df)

,run_id,snapshot_used,ok,src_rows,snap_rows,ts
0,20260514_144903,retention_actions_snapshot_20251101_20260508,True,316,316,2026-05-14 14:49:08.740333+00:00


---
## Bloque 6 · Auditoría + cierre

Tres queries que un DPO o auditor te pediría en una revisión de compliance.

In [30]:
# 6.1 — Inventario completo de snapshots en audit_archive
# (los nuestros de hoy + los que aparezcan tras varios días de cron)
snaps = list(bq.query(f"""
  SELECT
    table_name,
    base_table_schema,
    base_table_name,
    snapshot_time,
    TIMESTAMP_DIFF(CURRENT_TIMESTAMP(), snapshot_time, DAY) AS antiguedad_dias
  FROM `{PROJECT_ID}.{AUDIT}`.INFORMATION_SCHEMA.TABLE_SNAPSHOTS
  ORDER BY snapshot_time DESC
""").result())

print(f"Inventario {AUDIT}: {len(snaps)} snapshots")
for s in snaps[:15]:
    print(f"  {s.snapshot_time:%Y-%m-%d %H:%M} ({s.antiguedad_dias:>3}d) "
          f"{s.base_table_schema}.{s.base_table_name} -> {s.table_name}")

Inventario audit_archive: 1 snapshots
  2026-05-14 14:48 (  0d) scratch_main.demo_critical -> demo_critical_snap_20260514_144845


In [31]:
# 6.2 — Auditoría de operaciones sobre tablas críticas (último día)
audit = list(bq.query(f"""
  SELECT
    user_email,
    job_type,
    statement_type,
    creation_time,
    SUBSTR(query, 1, 100) AS query_preview
  FROM `region-{REGION}`.INFORMATION_SCHEMA.JOBS_BY_PROJECT
  WHERE creation_time >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 1 DAY)
    AND (
      REGEXP_CONTAINS(LOWER(IFNULL(query,'')), r'retention_actions|dim_employee|fact_salary')
      OR statement_type IN ('DROP_TABLE','ALTER_TABLE','CREATE_SNAPSHOT_TABLE')
    )
  ORDER BY creation_time DESC
  LIMIT 25
""").result())

print(f"Operaciones críticas últimas 24h: {len(audit)}")
for a in audit[:10]:
    print(f"  {a.creation_time:%H:%M} {a.user_email[:40]:<40} {a.statement_type or a.job_type:<25} {a.query_preview[:60]}")

Operaciones críticas últimas 24h: 25
  14:49 carlos.g.chou@gmail.com                  INSERT                    
  INSERT INTO `project-9176af0b-ecb3-4050-859.pipeline_runs
  14:49 carlos.g.chou@gmail.com                  SELECT                    SELECT COUNT(*) c FROM `project-9176af0b-ecb3-4050-859.scrat
  14:49 carlos.g.chou@gmail.com                  SELECT                    SELECT COUNT(*) c FROM `project-9176af0b-ecb3-4050-859.predi
  14:49 carlos.g.chou@gmail.com                  CREATE_TABLE              CREATE TABLE `project-9176af0b-ecb3-4050-859.scratch_restore
  14:49 carlos.g.chou@gmail.com                  SELECT                    
  SELECT table_name
  FROM `project-9176af0b-ecb3-4050-859.
  14:48 carlos.g.chou@gmail.com                  DROP_TABLE                DROP TABLE `project-9176af0b-ecb3-4050-859.scratch_main.demo
  14:48 carlos.g.chou@gmail.com                  CREATE_SNAPSHOT_TABLE     
  CREATE SNAPSHOT TABLE `project-9176af0b-ecb3-4050-859.aud
  14:48 car

In [32]:
# 6.3 — Checklist DPO-friendly (12 puntos)
# Marca lo que está cumplido sobre el repo y las tablas reales del proyecto.

repo_files = {p.relative_to(REPO).as_posix() for p in REPO.rglob("*") if p.is_file() and ".git/" not in str(p)}
def has(path): return path in repo_files

checks = {
    "Existe RETENTION.md versionada":             has("backups/policies/RETENTION.md"),
    "Existe CODEOWNERS con @dpo":                 has(".github/CODEOWNERS") and "@dpo" in (REPO/".github/CODEOWNERS").read_text(),
    "Existe workflow de validación CI":           has(".github/workflows/validate-backups.yml"),
    "Existe workflow de snapshot programado":     has(".github/workflows/scheduled-snapshot.yml"),
    "Existe check_retention_policy.py":           has("backups/scripts/check_retention_policy.py"),
    "Existe SQL de snapshot retention_actions":   has("backups/sql/snapshot_retention_actions.sql"),
    "Existe SQL de snapshot dim_employee":        has("backups/sql/snapshot_dim_employee.sql"),
    "Existe SQL de políticas retention":          has("backups/sql/retention_policies.sql"),
    "Existe export_to_gcs.py":                    has("backups/python/export_to_gcs.py"),
    "Existe restore_test.py":                     has("backups/python/restore_test.py"),
    "Hay al menos 1 snapshot en audit_archive":   len(snaps) >= 1,
    "Hay al menos 1 restore_test OK":             ok,
}
print("\n=== CHECKLIST DPO ===")
for k, v in checks.items():
    print(f"  {'✓' if v else '✗'}  {k}")

n_ok = sum(checks.values())
print(f"\n{n_ok}/{len(checks)} cumplidos")
if n_ok == len(checks):
    print("✓ Listo para auditoría")
else:
    print("⚠ Revisar los pendientes antes de la próxima auditoría")


=== CHECKLIST DPO ===
  ✓  Existe RETENTION.md versionada
  ✓  Existe CODEOWNERS con @dpo
  ✓  Existe workflow de validación CI
  ✓  Existe workflow de snapshot programado
  ✓  Existe check_retention_policy.py
  ✓  Existe SQL de snapshot retention_actions
  ✓  Existe SQL de snapshot dim_employee
  ✓  Existe SQL de políticas retention
  ✓  Existe export_to_gcs.py
  ✓  Existe restore_test.py
  ✓  Hay al menos 1 snapshot en audit_archive
  ✓  Hay al menos 1 restore_test OK

12/12 cumplidos
✓ Listo para auditoría


In [33]:
# 6.4 — (Opcional) push del repo a GitHub
# Si tienes gh CLI autenticada y quieres subirlo, descomenta:
#
# r = subprocess.run(["gh", "repo", "create", "people-analytics-ops",
#                     "--private", "--source", str(REPO), "--push"],
#                    capture_output=True, text=True)
# print(r.stdout); print(r.stderr)
#
# Lo dejamos comentado por defecto: el alumno decide si subirlo o no.
# Toda la actividad del notebook es completa SIN subir a GitHub.

print("Repo local en:", REPO)
print("Para subirlo cuando quieras:")
print("  gh repo create people-analytics-ops --private --source", REPO, "--push")

Repo local en: /Users/main/Desktop/curso-vertexai/sesion_05_actividad/repo_people_analytics_ops
Para subirlo cuando quieras:
  gh repo create people-analytics-ops --private --source /Users/main/Desktop/curso-vertexai/sesion_05_actividad/repo_people_analytics_ops --push


---
## Cierre

Lo que tienes ahora:

- Un repo local (`./repo_people_analytics_ops/`) con la estructura completa de Backup-as-Code
- Snapshots reales sobre `predictions_retention.retention_actions` (S3) en `audit_archive`
- Export Parquet+Snappy en `gs://<bucket>/backups/scratch/demo_critical/`
- 3 caminos de recuperación probados sobre tabla real con tiempos medidos
- Una fila en `pipeline_runs.restore_tests` que demuestra que el plan funciona

### Lo que el notebook NO hizo (porque no debe)

- Subir el repo a GitHub (decisión del alumno; comando documentado)
- Aplicar `retention_policies.sql` a las tablas reales (requiere DPO approval — el SQL queda en el repo, no se ejecuta)
- Borrar tus snapshots de S3 (siguen ahí, lo que añadimos es la **política**)

### Conexión con S6

La próxima sesión introduce Cloud DLP (M11). Los exports Parquet que generamos hoy en `gs://.../backups/` serán el **input** del scan DLP de la próxima sesión: detección automática de PII en respaldos antes de que crucen entornos.